In [6]:
import pandas as pd
import geopandas as gpd
from pathlib import Path

current_path = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in [current_path, *current_path.parents]
     if (p / "Scripts").is_dir()),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Run this notebook from within the repository.")
    
# Load data
# FIPS
fips_path = PROJECT_ROOT/"Data/Geography/FIPS/US states FIPS.csv"   # FIPS path
df_fips = pd.read_csv(fips_path)  # Load FIPS
df_fips["FIPS Code"] = df_fips["FIPS Code"].astype(str).str.zfill(2)    # Ensure string, add 0 to single digits

# MSAs
msa_shapefile_path = PROJECT_ROOT/"Data/Geography/CBSA_shapefile_2025/tl_2025_us_cbsa.shp"    # Shapefile path
gdf_msa = gpd.read_file(msa_shapefile_path) # Load MSA shapefile

gdf_msa = gdf_msa[gdf_msa["LSAD"] == 'M1'].reset_index(drop=True)   # Remove micro
gdf_msa['State_Abbr'] = gdf_msa['NAME'].str[-2:]    # Get state abbriviations
gdf_msa = gdf_msa[gdf_msa['State_Abbr'].isin(df_fips['Postal Abbr.'])].copy()   # Filter only US states
valid_geoid = gdf_msa["GEOID"].astype(str).tolist()  # GEOID list

# Lawyers
lawyers_path = PROJECT_ROOT/"Data/BrightData_Lawyers/BrightData_Lawyers_master_normalized_1overN.csv" # Lawyers data Path
lawyers_data = pd.read_csv(lawyers_path)   # Load data
lawyers_data.rename(columns={'CBSA': 'AREA'}, inplace=True) # Rename columns
lawyers_data["AREA"] = lawyers_data["AREA"].astype("Int64").astype(str)
lawyers_data = lawyers_data[lawyers_data["AREA"].isin(valid_geoid)]   # Take only MSAs from the list

# Proxies
# Patents
patents_path = PROJECT_ROOT/"Data/Proxies/Intellectual Property/Patents 2000-2015.csv" # Granted patents data Path
patents_data = pd.read_csv(patents_path)   # Load data

patents_data = patents_data[['ID Code', 'U.S. Regional Title', '2015']].copy()  # Take relevant columns
patents_data.rename(columns={'ID Code': 'AREA', 'U.S. Regional Title': 'MSA', '2015': 'Patents'}, inplace=True) # Rename columns

patents_data = patents_data.iloc[:-1].reset_index(drop=True) # Remove last row

patents_data["AREA"] = patents_data["AREA"].astype("Int64").astype(str) # GEOID as string
patents_data['AREA'] = patents_data['AREA'].str[-5:]    # Get GEOID

patents_data = patents_data[patents_data["AREA"].isin(valid_geoid)]   # Take only MSAs from the list

patents_data = patents_data.merge(lawyers_data[['AREA', 'Intellectual Property Law_normalized_1overN_count']], on=['AREA'], how='left')
patents_data = patents_data.dropna(subset=['Intellectual Property Law_normalized_1overN_count'])
patents_data = patents_data.rename(columns={"Intellectual Property Law_normalized_1overN_count": "Intellectual Property"})
patents_data = patents_data.drop(columns=["MSA", "MSA_Name", "msa_name"], errors="ignore")

patents_data.to_csv(PROJECT_ROOT/"Data/Proxies/Intellectual Property/Intellectual_Property_Proxy_Normalized.csv", index=False)